## alpha_tutorial — AlphaEarth K-Means Clustering Intro

**Purpose**: Introduction to AlphaEarth 64-dim satellite embeddings via K-means clustering.

**Inputs**: Google Earth Engine — `GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL`
**Outputs**: Interactive geemap visualization in-notebook (no files written)
**GEE auth required**: Yes

**Parameters**:
- Number of clusters: 3, 5, or 10
- Region: configurable AOI

**Expected runtime**: < 2 min (server-side EE computation)

**How to run**: Execute cells top-to-bottom after GEE authentication.


# AlphaEarth Embeddings Python Tutorial

## Clustering Example

This python was converted from the original javascript here:

https://developers.google.com/earth-engine/tutorials/community/satellite-embedding-01-introduction

In [ ]:
import ee
import geemap

# ee.Authenticate())
ee.Initialize(project='ardent-fusion-421917')

In [ ]:
# ── Parameters — edit these to change the demo location or clustering settings ─
DEMO_LON  = 76.4    # center longitude (default: Karnataka, India)
DEMO_LAT  = 12.45   # center latitude
DEMO_SPAN = 0.27    # bounding-box half-width in degrees (~30 km)
YEAR      = 2024    # AlphaEarth embedding year to load (2017–2024)
N_CLUSTERS_LIST = [3, 5, 10]   # K values to compare
N_SAMPLES = 1000    # pixels sampled for K-means training


Use the satellite basemap (Note: Map.setOptions is specific to Code Editor)

In Python, you'll typically use geemap or folium for visualization

In [ ]:
embeddings = ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL")

geometry = ee.Geometry.Rectangle([
    DEMO_LON - DEMO_SPAN, DEMO_LAT - DEMO_SPAN,
    DEMO_LON + DEMO_SPAN, DEMO_LAT + DEMO_SPAN,
])

In [ ]:
year = YEAR
start_date = ee.Date.fromYMD(year, 1, 1)
end_date = start_date.advance(1, "year")

filtered_embeddings = embeddings.filter(ee.Filter.date(start_date, end_date)).filter(
    ee.Filter.bounds(geometry)
)

In [ ]:
embeddings_image = filtered_embeddings.mosaic()
print("Satellite Embedding Image", embeddings_image.getInfo())

In [ ]:
n_samples = N_SAMPLES
training = embeddings_image.sample(
    region=geometry, scale=10, numPixels=n_samples, seed=100
)
print(training.first().getInfo())

In [ ]:
# Function to train a model for desired number of clusters
def get_clusters(n_clusters):
    clusterer = ee.Clusterer.wekaKMeans(n_clusters).train(training)

    # Cluster the image
    clustered = embeddings_image.cluster(clusterer)
    return clustered


# run K-means for each value in N_CLUSTERS_LIST
clusters = {n: get_clusters(n) for n in N_CLUSTERS_LIST}
cluster3  = clusters.get(3,  get_clusters(3))
cluster5  = clusters.get(5,  get_clusters(5))
cluster10 = clusters.get(10, get_clusters(10))

To visualize the results in Python, you can use geemap:

In [ ]:
vis_params = {"min": -0.3, "max": 0.3, "bands": ["A01", "A16", "A09"]}

Map = geemap.Map()
Map.centerObject(geometry, 12)
Map.addLayer(embeddings_image.clip(geometry), vis_params, 'Embeddings Image')
Map.addLayer(cluster3.randomVisualizer().clip(geometry), {}, '3 clusters')
Map.addLayer(cluster5.randomVisualizer().clip(geometry), {}, '5 clusters')
Map.addLayer(cluster10.randomVisualizer().clip(geometry), {}, '10 clusters')
Map